# Fine-Tuning GPT-OSS 20B with Unsloth → TensorRT-LLM Deployment

Fine-tune GPT-OSS 20B for research paper Q&A using the **QASPER dataset**, then deploy with TensorRT-LLM on Blackwell.

**Platform**: Thinkube on DGX Spark (Blackwell GB10)

**⚠️ Requirements**: 
- Run this notebook with `tk-jupyter-fine-tuning` image
- Blackwell GPU required

**Prerequisites**: Complete `03-multi-agent.ipynb` first.

## Why QASPER?

[QASPER](https://huggingface.co/datasets/allenai/qasper) (Question Answering on Scientific Papers) is a dataset of 5,049 questions over 1,585 NLP research papers. Each question was written by an NLP practitioner and answered with evidence from the full paper text.

| Property | Value |
|----------|-------|
| Size | 5,049 Q&A pairs |
| Domain | NLP research papers |
| License | CC-BY-4.0 |
| Source | Human-annotated (not model-generated) |

This is ideal for fine-tuning because:
1. **Human-created** - Not generated by the model we're training (avoids circular training)
2. **Domain-specific** - NLP papers match our Research Assistant use case
3. **Evidence-grounded** - Answers include supporting spans from papers
4. **Permissive license** - CC-BY-4.0 allows commercial use

## References

This notebook adapts the official Unsloth GPT-OSS 20B fine-tuning notebook for our use case:

- **[Unsloth GPT-OSS 20B Fine-tuning (Official)](https://github.com/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-Fine-tuning.ipynb)** - Source for model loading, LoRA config, and training parameters
- [QASPER Dataset (HuggingFace)](https://huggingface.co/datasets/allenai/qasper)
- [QASPER Paper (arXiv)](https://arxiv.org/abs/2105.03011)
- [Train LLM on Blackwell with Unsloth (NVIDIA Blog)](https://developer.nvidia.com/blog/train-an-llm-on-an-nvidia-blackwell-desktop-with-unsloth-and-scale-it/)
- [TensorRT-LLM Python SDK](https://github.com/NVIDIA/TensorRT-LLM)

## The Complete Workflow

```
┌─────────────────────────────────────────────────────────────────────────┐
│           Unsloth Fine-Tuning → TensorRT-LLM Deployment                 │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  1. LOAD DATA   QASPER dataset from HuggingFace                        │
│       ↓                                                                 │
│  2. FORMAT      Convert to instruction-tuning format                   │
│       ↓                                                                 │
│  3. FINE-TUNE   GPT-OSS 20B + Unsloth (2x faster, 60% less memory)     │
│       ↓                                                                 │
│  4. TRACK       MLflow experiment logging                              │
│       ↓                                                                 │
│  5. EVALUATE    Compare base vs fine-tuned on research Q&A             │
│       ↓                                                                 │
│  6. EXPORT      Merge LoRA → HuggingFace checkpoint                    │
│       ↓                                                                 │
│  7. CONVERT     HuggingFace → TensorRT-LLM engine                      │
│       ↓                                                                 │
│  8. DEPLOY      tkt-tensorrt-llm template + LiteLLM registration       │
│                                                                         │
│  Result: Fine-tuned model running at TensorRT-LLM speeds               │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

## Setup

Check GPU availability and connect to MLflow.

In [ ]:
import os
import torch

# Check GPU - should be Blackwell
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"Memory: {gpu_mem:.1f} GB")
    
    # Check for Blackwell
    if "Blackwell" in gpu_name or "GB" in gpu_name:
        print("✓ Blackwell GPU detected - optimal for Unsloth + TensorRT-LLM")

# MLflow for experiment tracking
MLFLOW_TRACKING_URI = os.environ.get('MLFLOW_TRACKING_URI')
print(f"\nMLflow: {MLFLOW_TRACKING_URI}")

# Unsloth recommended sequence length for GPT-OSS 20B
max_seq_length = 1024
dtype = None  # Auto-detect (bfloat16 on Blackwell)

---
## 1. Load QASPER Dataset

Load the QASPER dataset from HuggingFace - 5,049 Q&A pairs on NLP research papers.

In [ ]:
from datasets import load_dataset

# Load QASPER dataset
# - train: 2,593 papers with Q&A
# - validation: 506 papers  
# - test: 486 papers
dataset = load_dataset("allenai/qasper")

print(f"Train papers: {len(dataset['train'])}")
print(f"Validation papers: {len(dataset['validation'])}")
print(f"Test papers: {len(dataset['test'])}")

# Explore structure of one example
example = dataset['train'][0]
print(f"\nExample paper: {example['title']}")
print(f"Number of Q&A pairs: {len(example['qas']['question'])}")

---
## 2. Format for Instruction Tuning

Convert QASPER's nested structure into instruction-tuning format compatible with Unsloth/TRL.

In [ ]:
def extract_qasper_examples(dataset_split):
    """
    Convert QASPER nested format to chat message format.
    
    QASPER structure:
    - Each row is a paper with title, abstract, full_text, and qas
    - qas contains multiple questions, each with multiple annotator answers
    
    Output format for chat template:
    {
        "messages": [
            {"role": "user", "content": "..."},
            {"role": "assistant", "content": "..."}
        ]
    }
    """
    examples = []
    
    for paper in dataset_split:
        title = paper['title']
        abstract = paper['abstract']
        
        # Extract each Q&A pair
        for i, question in enumerate(paper['qas']['question']):
            # Get answers from all annotators
            answers_data = paper['qas']['answers'][i]
            
            # Find first non-empty, answerable response
            answer_text = None
            for annotator_answers in answers_data['answer']:
                # Skip unanswerable questions
                if annotator_answers.get('unanswerable', False):
                    continue
                    
                # Try extractive spans first (direct quotes from paper)
                if annotator_answers.get('extractive_spans'):
                    answer_text = " ".join(annotator_answers['extractive_spans'])
                    break
                # Fall back to free-form answer
                elif annotator_answers.get('free_form_answer'):
                    answer_text = annotator_answers['free_form_answer']
                    break
                # Yes/no answers
                elif annotator_answers.get('yes_no') is not None:
                    answer_text = "Yes" if annotator_answers['yes_no'] else "No"
                    break
            
            # Skip if no valid answer found
            if not answer_text:
                continue
            
            # Format as chat messages (official Unsloth pattern)
            user_content = f"""Paper: {title}

Abstract: {abstract}

Question: {question}"""
            
            examples.append({
                "messages": [
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": answer_text}
                ]
            })
    
    return examples

# Convert training data
train_examples = extract_qasper_examples(dataset['train'])
val_examples = extract_qasper_examples(dataset['validation'])

print(f"Training examples: {len(train_examples)}")
print(f"Validation examples: {len(val_examples)}")

# Show example
print("\n" + "="*60)
print("EXAMPLE:")
print("="*60)
ex = train_examples[0]
print(f"USER: {ex['messages'][0]['content'][:500]}...")
print(f"\nASSISTANT: {ex['messages'][1]['content']}")

In [ ]:
# Convert to HuggingFace Dataset format for training
from datasets import Dataset

train_dataset = Dataset.from_list(train_examples)
val_dataset = Dataset.from_list(val_examples)

print(f"Train dataset: {train_dataset}")
print(f"Validation dataset: {val_dataset}")

# Preview what the chat template produces
print("\n" + "="*60)
print("CHAT TEMPLATE PREVIEW:")
print("="*60)
# Note: tokenizer not loaded yet, this will be shown after model loading

---
## 3. Load Model with Unsloth

Unsloth optimizations for Blackwell:
- 2x faster training
- 60% less memory
- Native 4-bit quantization
- Optimized LoRA kernels

In [ ]:
from unsloth import FastLanguageModel

# Load GPT-OSS 20B with Unsloth optimizations
# Using Unsloth's optimized version of the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gpt-oss-20b",  # Unsloth's optimized version
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
    full_finetuning=False,  # LoRA, not full fine-tuning
)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Max sequence length: {max_seq_length}")

In [ ]:
# Preview what the chat template produces
sample = train_examples[0]
formatted = tokenizer.apply_chat_template(
    sample["messages"],
    tokenize=False,
    add_generation_prompt=False,
)
print("CHAT TEMPLATE OUTPUT:")
print("="*60)
print(formatted[:1000])
print("...")

In [ ]:
# Add LoRA adapters for parameter-efficient fine-tuning
# Parameters from official Unsloth GPT-OSS 20B notebook
model = FastLanguageModel.get_peft_model(
    model,
    r=8,               # LoRA rank (Unsloth default for GPT-OSS)
    target_modules=[   # Which layers to adapt
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
        "gate_proj", "up_proj", "down_proj"       # MLP
    ],
    lora_alpha=16,
    lora_dropout=0,    # No dropout - Unsloth recommendation
    bias="none",       # No bias training
    use_gradient_checkpointing="unsloth",  # Unsloth-optimized checkpointing
    random_state=3407,  # Unsloth's default seed
    use_rslora=False,
    loftq_config=None,
)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} ({100*trainable/total:.2f}%)")

---
## 4. Train with MLflow Tracking

Fine-tune with full experiment tracking:
- Hyperparameters logged
- Loss curves tracked
- Checkpoints saved to MLflow Model Registry

In [ ]:
import mlflow
from trl import SFTConfig, SFTTrainer

# Connect to MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("research-assistant-fine-tuning")

# Format function using the model's native chat template (official Unsloth pattern)
def format_chat(example):
    """Apply the model's chat template to messages."""
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,  # Don't add prompt - we have the assistant response
    )

# Start MLflow run
with mlflow.start_run(run_name="gpt-oss-20b-qasper"):
    # Log hyperparameters (matching Unsloth notebook)
    mlflow.log_params({
        "base_model": "unsloth/gpt-oss-20b",
        "dataset": "allenai/qasper",
        "lora_r": 8,
        "lora_alpha": 16,
        "learning_rate": 2e-4,
        "max_steps": 100,  # Increase for real training
        "batch_size": 1,
        "gradient_accumulation": 4,
        "max_seq_length": max_seq_length,
    })
    
    # Configure trainer using SFTConfig (Unsloth pattern)
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        formatting_func=format_chat,
        max_seq_length=max_seq_length,
        args=SFTConfig(
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            warmup_steps=5,
            max_steps=100,  # For demo; increase for real training
            learning_rate=2e-4,
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.001,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir="outputs",
            report_to="none",  # We use MLflow instead
        ),
    )
    
    # Train
    print("Starting training...")
    trainer_stats = trainer.train()
    
    # Log metrics
    mlflow.log_metrics({
        "final_loss": trainer_stats.training_loss,
        "total_steps": trainer_stats.global_step,
        "train_samples": len(train_dataset),
    })
    
    print(f"\n✓ Training complete!")
    print(f"  Final loss: {trainer_stats.training_loss:.4f}")
    print(f"  Total steps: {trainer_stats.global_step}")

---
## 5. Evaluate Fine-Tuned Model

Compare base vs fine-tuned on research paper questions from the test set.

In [ ]:
# Enable fast inference mode
FastLanguageModel.for_inference(model)

# Test on examples from validation set using proper chat template
from transformers import TextStreamer

test_examples = val_examples[:3]

for ex in test_examples:
    # Use only the user message for inference
    messages = [ex["messages"][0]]  # Just the user question
    
    # Apply chat template (official Unsloth pattern)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # Add prompt for assistant to respond
    )
    
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    
    print("="*60)
    print(f"QUESTION: {ex['messages'][0]['content'].split('Question:')[-1].strip()}")
    print(f"\nEXPECTED: {ex['messages'][1]['content']}")
    print(f"\nMODEL:")
    
    # Stream output (official Unsloth pattern)
    streamer = TextStreamer(tokenizer, skip_prompt=True)
    _ = model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
    )
    print()

---
## 6. Export to HuggingFace Format

Merge LoRA adapters and save as full HuggingFace checkpoint for TensorRT-LLM conversion.

In [ ]:
# Merge LoRA adapters into base model and save
# Following Unsloth's save pattern
OUTPUT_DIR = "gpt-oss-20b-qasper"

# Save merged 16-bit model (required for TensorRT-LLM conversion)
model.save_pretrained_merged(
    OUTPUT_DIR,
    tokenizer,
    save_method="merged_16bit",
)

print(f"✓ Merged model saved to: {OUTPUT_DIR}/")

# Also save to MLflow Model Registry for tracking
with mlflow.start_run(run_name="gpt-oss-20b-qasper-export"):
    mlflow.log_artifacts(OUTPUT_DIR, artifact_path="hf-checkpoint")
    
    # Register model
    mlflow.register_model(
        f"runs:/{mlflow.active_run().info.run_id}/hf-checkpoint",
        "gpt-oss-20b-qasper"
    )

print("✓ Model registered in MLflow Model Registry")

---
## 7. Convert to TensorRT-LLM Engine

Build optimized TensorRT-LLM engine for Blackwell inference.

This uses NVIDIA's TensorRT-LLM Python SDK.

In [ ]:
# Convert HuggingFace checkpoint to TensorRT-LLM engine
# 
# Note: This step requires the TensorRT-LLM container/environment.
# In JupyterHub, use the tk-jupyter-fine-tuning image which includes TensorRT-LLM.

from tensorrt_llm import LLM, BuildConfig

# Configure build for Blackwell
build_config = BuildConfig(
    max_batch_size=8,
    max_input_len=2048,
    max_seq_len=4096,
)

# Build TensorRT engine
print("Building TensorRT-LLM engine (this may take several minutes)...")
llm = LLM(
    model=OUTPUT_DIR,  # Our merged HuggingFace checkpoint
    build_config=build_config,
)

# Save engine
TRT_OUTPUT_DIR = "gpt-oss-20b-qasper-trt"
llm.save(TRT_OUTPUT_DIR)

print(f"✓ TensorRT-LLM engine saved to: {TRT_OUTPUT_DIR}/")

---
## 8. Deploy with tkt-tensorrt-llm

Upload the TensorRT engine to MLflow Model Registry, then deploy via Thinkube Control.

In [ ]:
# Upload TensorRT engine to MLflow
with mlflow.start_run(run_name="gpt-oss-20b-qasper-tensorrt"):
    mlflow.log_artifacts(TRT_OUTPUT_DIR, artifact_path="tensorrt-engine")
    
    # Register model
    mlflow.register_model(
        f"runs:/{mlflow.active_run().info.run_id}/tensorrt-engine",
        "gpt-oss-20b-qasper-tensorrt"
    )

print("✓ TensorRT engine uploaded to MLflow Model Registry")
print("")
print("Next steps:")
print("1. Open Thinkube Control → Model Catalog")
print("2. Find 'gpt-oss-20b-qasper-tensorrt' in the registry")
print("3. Deploy with tkt-tensorrt-llm template")
print("4. Wait for pod to become healthy")

---
## 9. Register in LiteLLM

Add the fine-tuned model to LiteLLM gateway so it's accessible via the unified API.

In [ ]:
import requests

LITELLM_ENDPOINT = os.environ.get('LITELLM_ENDPOINT')
LITELLM_MASTER_KEY = os.environ.get('LITELLM_MASTER_KEY')

# Update these to match your deployment
FINETUNED_SERVICE_URL = "http://gpt-oss-qasper.default.svc:8355"
FINETUNED_MODEL_NAME = "gpt-oss-qasper"

config = {
    "model_name": FINETUNED_MODEL_NAME,
    "litellm_params": {
        "model": f"openai/{FINETUNED_MODEL_NAME}",
        "api_base": f"{FINETUNED_SERVICE_URL}/v1",
        "api_key": "not-needed"
    },
    "model_info": {
        "description": "GPT-OSS 20B fine-tuned on QASPER for research paper Q&A",
        "mode": "chat",
        "base_model": "openai-community/gpt-oss-20b",
        "fine_tuned": True,
        "dataset": "allenai/qasper"
    }
}

resp = requests.post(
    f"{LITELLM_ENDPOINT}/model/new",
    headers={
        "Authorization": f"Bearer {LITELLM_MASTER_KEY}",
        "Content-Type": "application/json"
    },
    json=config,
    timeout=30
)

if resp.status_code == 200:
    print(f"✓ Registered: {FINETUNED_MODEL_NAME}")
elif resp.status_code == 400 and "already exists" in resp.text.lower():
    print(f"ℹ Model {FINETUNED_MODEL_NAME} already registered")
else:
    print(f"✗ Failed: {resp.status_code} - {resp.text}")

---
## 10. Test Fine-Tuned Model via LiteLLM

Compare the base model vs fine-tuned model on the same research questions.

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=LITELLM_ENDPOINT,
    api_key=LITELLM_MASTER_KEY
)

# Test question
test_question = """Paper: Attention Is All You Need

Abstract: The dominant sequence transduction models are based on complex recurrent or 
convolutional neural networks that include an encoder and a decoder. The best performing 
models also connect the encoder and decoder through an attention mechanism. We propose 
a new simple network architecture, the Transformer, based solely on attention mechanisms, 
dispensing with recurrence and convolutions entirely.

Question: What is the main contribution of this paper?"""

print("="*60)
print("BASE MODEL (gpt-oss)")
print("="*60)
base_response = client.chat.completions.create(
    model="gpt-oss",
    messages=[{"role": "user", "content": test_question}],
    max_tokens=200
)
print(base_response.choices[0].message.content)

print("\n" + "="*60)
print("FINE-TUNED MODEL (gpt-oss-qasper)")
print("="*60)
ft_response = client.chat.completions.create(
    model="gpt-oss-qasper",
    messages=[{"role": "user", "content": test_question}],
    max_tokens=200
)
print(ft_response.choices[0].message.content)

---
## Summary

You've completed the full fine-tuning → deployment pipeline:

| Step | Tool | Output |
|------|------|--------|
| Dataset | QASPER (HuggingFace) | 5,049 research Q&A pairs |
| Fine-tune | Unsloth | LoRA adapters |
| Track | MLflow | Experiment logs |
| Export | Unsloth | HuggingFace checkpoint |
| Convert | TensorRT-LLM | Optimized engine |
| Deploy | tkt-tensorrt-llm | Running service |
| Gateway | LiteLLM | Unified API |

### What We Built

**Before**: Generic GPT-OSS 20B model
```python
client.chat.completions.create(model="gpt-oss", messages=[...])
```

**After**: Research paper Q&A specialist
```python
client.chat.completions.create(model="gpt-oss-qasper", messages=[...])
```

### Why This Matters

1. **Real training data** - QASPER is human-annotated, not model-generated
2. **Domain expertise** - Fine-tuned specifically for NLP research paper Q&A
3. **Production speed** - TensorRT-LLM optimization for Blackwell
4. **Unified API** - Same LiteLLM endpoint as all other models

### Key Learnings

- **LoRA/Unsloth** - Fine-tune 20B parameter model on single GPU
- **MLflow** - Track experiments, register models
- **TensorRT-LLM** - Convert HuggingFace → optimized inference
- **LiteLLM** - Unified gateway for all models (base and fine-tuned)

---

**Platform**: Thinkube - 100% self-contained AI development on DGX Spark (Blackwell)

**Dataset**: [QASPER](https://huggingface.co/datasets/allenai/qasper) - CC-BY-4.0 licensed research paper Q&A

**Workflow**: Unsloth fine-tuning → TensorRT-LLM deployment → LiteLLM gateway